# EgyMonuments.com Scraper



In [2]:
!pip install selenium


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException


## Step 1

In [3]:
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
# options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

all_sites = []

driver.get("https://egymonuments.com/locations")

wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#portfolio > div > div > div:nth-child(1)")))
time.sleep(1)

i = 1
while True:
    try:
        item = driver.find_element(By.CSS_SELECTOR, f"#portfolio > div > div > div:nth-child({i})")
    except NoSuchElementException:
        print("Finished — no more items.")
        break

    try:
        place_name = item.find_element(By.CSS_SELECTOR, "div.p-2.w-100").text.strip()
    except NoSuchElementException:
        place_name = ""

    try:
        place_location = item.find_element(By.CSS_SELECTOR, "div.itemlocation").text.strip()
    except NoSuchElementException:
        place_location = ""

    try:
        figure = item.find_element(By.TAG_NAME, "figure")
        style = figure.get_attribute("style") or ""
        match = re.search(r"url\(['\"]?(.*?)['\"]?\)", style)
        photo_url = match.group(1) if match else ""
    except NoSuchElementException:
        photo_url = ""

    try:
        detail_link = item.find_element(By.TAG_NAME, "a").get_attribute("href")
    except NoSuchElementException:
        detail_link = ""

    print(i, "-", place_name, "|", place_location, "|", detail_link)

    all_sites.append({
        "place_name": place_name,
        "place_location": place_location,
        "photo_url": photo_url,
        "detail_link": detail_link,
    })

    i += 1

print("all_sites:", len(all_sites))


1 - Giza plateau | Giza | https://egymonuments.com/details/Pyramids
2 - Egyptian Museum | Cairo | https://egymonuments.com/details/EgyptianMuseum
3 - Sharm El Sheikh Museum | Sharm El Sheikh | https://egymonuments.com/details/SharmElSheikhMuseum
4 - Hurghada Museum | Hurghada | https://egymonuments.com/details/HurghadaMuseum
5 - Luxor Temple | Luxor | https://egymonuments.com/details/LuxorTemple
6 - Karnak Temple | Luxor | https://egymonuments.com/details/KarnakTemple
7 - Hatshepsut Temple | Luxor | https://egymonuments.com/details/HatshepsutTemple
8 - Abu Simbel Temple | Abu Simbel | https://egymonuments.com/details/AbuSimbelTemple
9 - Coptic Museum | Cairo | https://egymonuments.com/details/CopticMuseum
10 - Dandarah Temple | Qena | https://egymonuments.com/details/DandarahTemple
11 - Deir El Madina | Luxor | https://egymonuments.com/details/DeirElMadinaTemple
12 - Valley Of Kings | Luxor | https://egymonuments.com/details/ValleyOfKings
13 - Luxor Museum | Luxor | https://egymonument

In [4]:
df = pd.DataFrame(all_sites)
df


,place_name,place_location,photo_url,detail_link
0,Giza plateau,Giza,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/Pyramids
1,Egyptian Museum,Cairo,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/EgyptianMuseum
2,Sharm El Sheikh Museum,Sharm El Sheikh,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/SharmElSheikh...
3,Hurghada Museum,Hurghada,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/HurghadaMuseum
4,Luxor Temple,Luxor,https://egymonuments.com/storage/events/Septem...,https://egymonuments.com/details/LuxorTemple
...,...,...,...,...
89,Kafr al-Sheikh Museum,kafr el-shiekh,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/KafrElshaikh
90,Tanis (San al-Hagar),Al-Sharqia,https://egymonuments.com/storage/events/Septem...,https://egymonuments.com/details/Tanis%28Sanal...
91,Grand Egyptian Museum,Giza,https://egymonuments.com/storage/events/August...,https://visit-gem.com/
92,Temple of Aghurmi in Siwa,Siwa,https://egymonuments.com/storage/events/Januar...,https://egymonuments.com/details/TempleofAghur...


In [8]:
df["photo_url"][70]

'https://egymonuments.com/storage/events/July2024/8FxIOT4BiX273eJrG7Zb.png'

## Step 2 

In [9]:
def extract_prices(container_selector):
    rows_out = []
    rows = driver.find_elements(By.CSS_SELECTOR, f"{container_selector} table tbody tr")
    for r in rows:
        tds = r.find_elements(By.TAG_NAME, "td")
        if len(tds) == 2:
            label = tds[0].text.strip()
            value = tds[1].text.strip().replace("\n", " ")
            if label:
                rows_out.append(f"{label}: {value}")
    return "\n".join(rows_out)


for idx, row in df.iterrows():
    link = row["detail_link"]
    if not link:
        continue

    print(f"[{idx + 1}/{len(df)}] Opening: {row['place_name']}")
    driver.get(link)

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#pills-OtherNationality")))
        tickets_price_other = extract_prices("#pills-OtherNationality") or "Free"
    except (NoSuchElementException, TimeoutException):
        tickets_price_other = "Free"

    tickets_price_egyptian = "Not Available"
    try:
        egy_tab_btn = driver.find_element(By.CSS_SELECTOR, "#pills-Egyption-tab")
        driver.execute_script("arguments[0].click();", egy_tab_btn)
        wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "#pills-Egyption")))
        tickets_price_egyptian = extract_prices("#pills-Egyption") or "Free"
    except NoSuchElementException:
        tickets_price_egyptian = "Not Available"

    try:
        hours_div = driver.find_element(By.XPATH, "//h3[contains(., 'OPENING HOURS')]/parent::div")
        opening_hours = hours_div.text.replace("OPENING HOURS", "").strip()
    except NoSuchElementException:
        opening_hours = "Not Available"

    try:
        booking_link = driver.find_element(By.CSS_SELECTOR, "a.btn-get-download").get_attribute("href")
    except NoSuchElementException:
        booking_link = ""

    # ---------- FREE ENTRY POLICY ----------
    free_entry_policy = "Not Available"
    try:
        cards = driver.find_elements(By.CSS_SELECTOR, "div.card.sdSlider")
        for card in cards:
            title_elem = card.find_element(By.CSS_SELECTOR, "h4.card-title")
            title_txt = driver.execute_script("return arguments[0].textContent;", title_elem).strip()
            if "FREE ENTRY POLICY" in title_txt.upper():
                text_elem = card.find_element(By.CSS_SELECTOR, "p.card-text")
                free_entry_policy = driver.execute_script("return arguments[0].textContent;", text_elem).strip()
                break
    except NoSuchElementException:
        pass

    df.at[idx, "tickets_price_other_nationality"] = tickets_price_other
    df.at[idx, "tickets_price_egyptian"] = tickets_price_egyptian
    df.at[idx, "opening_hours"] = opening_hours
    df.at[idx, "booking_link"] = booking_link
    df.at[idx, "free_entry_policy"] = free_entry_policy

driver.quit()
print("Done.")


[1/94] Opening: Giza plateau
[2/94] Opening: Egyptian Museum
[3/94] Opening: Sharm El Sheikh Museum
[4/94] Opening: Hurghada Museum
[5/94] Opening: Luxor Temple
[6/94] Opening: Karnak Temple
[7/94] Opening: Hatshepsut Temple
[8/94] Opening: Abu Simbel Temple
[9/94] Opening: Coptic Museum
[10/94] Opening: Dandarah Temple
[11/94] Opening: Deir El Madina
[12/94] Opening: Valley Of Kings
[13/94] Opening: Luxor Museum
[14/94] Opening: Esna Temple
[15/94] Opening: Salah Eldin Citadel
[16/94] Opening: Medinet Habu
[17/94] Opening: Philae Temple
[18/94] Opening: Kom Ombo Temple
[19/94] Opening: Mummification museum
[20/94] Opening: Ramesseum
[21/94] Opening: Saqqara Monuments
[22/94] Opening: The Temple Of Horus
[23/94] Opening: Tombs Of The Nobles
[24/94] Opening: Unfinished Obelisk
[25/94] Opening: Valley Of The Queens
[26/94] Opening: Islamic art museum
[27/94] Opening: Baron Empain Palace
[28/94] Opening: Nubia Museum
[29/94] Opening: Serapeum of Alexandria
[30/94] Opening: Catacombs
[31/9

In [11]:
df["booking_link"][93]

'https://egymonuments.com/book-date/259'

In [ ]:
df.to_csv("egymonuments_tickets.csv", index=False)


## Step 3 `on_map`

In [6]:
#import csv into df
df = pd.read_csv("egymonuments_tickets.csv")

In [7]:
df.head(10)

,place_name,place_location,photo_url,detail_link,tickets_price_other_nationality,tickets_price_egyptian,opening_hours,booking_link,free_entry_policy
0,Giza plateau,Giza,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/Pyramids,Adult: EGP 700\nStudent: EGP 350,Adult: EGP 60\nStudent: EGP 30,All Days\nSummer Working Hours:\nfrom 07:00 am...,https://egymonuments.com/book-date/2,Free entry for children below 6 years. ...
1,Egyptian Museum,Cairo,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/EgyptianMuseum,Adult: EGP 550\nStudent: EGP 275,Adult: EGP 30\nStudent: EGP 10,All Days\nSummer Working Hours:\nfrom 09:00 am...,https://egymonuments.com/book-date/3,Free entry for children below 6 years. ...
2,Sharm El Sheikh Museum,Sharm El Sheikh,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/SharmElSheikh...,Adult: EGP 200\nStudent: EGP 100,Adult: EGP 40\nStudent: EGP 20,All Days\nSummer Working Hours:\nfrom 10:00 am...,https://egymonuments.com/book-date/4,Free entry for children below 6 years. ...
3,Hurghada Museum,Hurghada,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/HurghadaMuseum,Adult: EGP 300\nStudent: EGP 150,Adult: EGP 80\nStudent: EGP 40,All Days\nSummer Working Hours:\nfrom 10:00 am...,https://egymonuments.com/book-date/5,Free entry for children below 6 years. ...
4,Luxor Temple,Luxor,https://egymonuments.com/storage/events/Septem...,https://egymonuments.com/details/LuxorTemple,Adult: EGP 500\nStudent: EGP 250,Adult: EGP 40\nStudent: EGP 20,All Days\nSummer Working Hours:\nfrom 06:00 am...,https://egymonuments.com/book-date/6,Free entry for children below 6 years. ...
5,Karnak Temple,Luxor,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/KarnakTemple,Adult: EGP 600\nStudent: EGP 300,Adult: EGP 40\nStudent: EGP 20,All Days\nSummer Working Hours:\nfrom 06:00 am...,https://egymonuments.com/book-date/7,Free entry for children below 6 years. ...
6,Hatshepsut Temple,Luxor,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/HatshepsutTemple,Adult: EGP 440\nStudent: EGP 220,Adult: EGP 40\nStudent: EGP 20,All Days\nSummer Working Hours:\nfrom 06:00 am...,https://egymonuments.com/book-date/8,Free entry for children below 6 years. ...
7,Abu Simbel Temple,Abu Simbel,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/AbuSimbelTemple,Adult: EGP 822\nStudent: EGP 445.5\nAdult: EGP...,Adult: EGP 30.5\nStudent: EGP 10.5\nAdult: EGP...,All Days\nSummer Working Hours:\nfrom 06:00 am...,https://egymonuments.com/book-date/9,Free entry for children below 6 years. ...
8,Coptic Museum,Cairo,https://egymonuments.com/storage/events/Septem...,https://egymonuments.com/details/CopticMuseum,Adult: EGP 280\nStudent: EGP 140,Adult: EGP 20\nStudent: EGP 10,All Days\nSummer Working Hours:\nfrom 09:00 am...,https://egymonuments.com/book-date/10,Free entry for children below 6 years. ...
9,Dandarah Temple,Qena,https://egymonuments.com/storage/events/Septem...,https://egymonuments.com/details/DandarahTemple,Adult: EGP 300\nStudent: EGP 150,Adult: EGP 20\nStudent: EGP 10,All Days\nSummer Working Hours:\nfrom 07:00 am...,https://egymonuments.com/book-date/11,Free entry for children below 6 years. ...


In [8]:
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

for idx, row in df.iterrows():
    query = f"{row['place_name']}, {row['place_location']}, Egypt"
    print("Searching Maps for:", query)

    driver.get("https://www.google.com/maps/search/" + query.replace(" ", "+") + "?hl=en")

    try:
        WebDriverWait(driver, 15).until(
            lambda d: d.find_elements(By.CSS_SELECTOR, "h1.DUwDvf")
            or d.find_elements(By.CSS_SELECTOR, "div.Nv2PK")
        )
    except:
        pass

    if driver.find_elements(By.CSS_SELECTOR, "div.Nv2PK"):
        try:
            first_result = driver.find_element(By.CSS_SELECTOR, "div.Nv2PK a.hfpxzc")
            first_result.click()
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1.DUwDvf"))
            )
        except:
            pass

    time.sleep(1.5)

    df.at[idx, "on_map"] = driver.current_url

driver.quit()

Searching Maps for: Giza plateau, Giza, Egypt
Searching Maps for: Egyptian Museum, Cairo, Egypt
Searching Maps for: Sharm El Sheikh Museum, Sharm El Sheikh, Egypt
Searching Maps for: Hurghada Museum, Hurghada, Egypt
Searching Maps for: Luxor Temple, Luxor, Egypt
Searching Maps for: Karnak Temple, Luxor, Egypt
Searching Maps for: Hatshepsut Temple, Luxor, Egypt
Searching Maps for: Abu Simbel Temple, Abu Simbel, Egypt
Searching Maps for: Coptic Museum, Cairo, Egypt
Searching Maps for: Dandarah Temple, Qena, Egypt
Searching Maps for: Deir El Madina, Luxor, Egypt
Searching Maps for: Valley Of Kings, Luxor, Egypt
Searching Maps for: Luxor Museum, Luxor, Egypt
Searching Maps for: Esna Temple, Esna, Egypt
Searching Maps for: Salah Eldin Citadel, Cairo, Egypt
Searching Maps for: Medinet Habu, Luxor, Egypt
Searching Maps for: Philae Temple, Aswan, Egypt
Searching Maps for: Kom Ombo Temple, Aswan, Egypt
Searching Maps for: Mummification museum, Luxor, Egypt
Searching Maps for: Ramesseum, Luxor, 

In [9]:
df

,place_name,place_location,photo_url,detail_link,tickets_price_other_nationality,tickets_price_egyptian,opening_hours,booking_link,free_entry_policy,on_map
0,Giza plateau,Giza,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/Pyramids,Adult: EGP 700\nStudent: EGP 350,Adult: EGP 60\nStudent: EGP 30,All Days\nSummer Working Hours:\nfrom 07:00 am...,https://egymonuments.com/book-date/2,Free entry for children below 6 years. ...,https://www.google.com/maps/search/Giza+platea...
1,Egyptian Museum,Cairo,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/EgyptianMuseum,Adult: EGP 550\nStudent: EGP 275,Adult: EGP 30\nStudent: EGP 10,All Days\nSummer Working Hours:\nfrom 09:00 am...,https://egymonuments.com/book-date/3,Free entry for children below 6 years. ...,https://www.google.com/maps/search/Egyptian+Mu...
2,Sharm El Sheikh Museum,Sharm El Sheikh,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/SharmElSheikh...,Adult: EGP 200\nStudent: EGP 100,Adult: EGP 40\nStudent: EGP 20,All Days\nSummer Working Hours:\nfrom 10:00 am...,https://egymonuments.com/book-date/4,Free entry for children below 6 years. ...,https://www.google.com/maps/search/Sharm+El+Sh...
3,Hurghada Museum,Hurghada,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/HurghadaMuseum,Adult: EGP 300\nStudent: EGP 150,Adult: EGP 80\nStudent: EGP 40,All Days\nSummer Working Hours:\nfrom 10:00 am...,https://egymonuments.com/book-date/5,Free entry for children below 6 years. ...,https://www.google.com/maps/search/Hurghada+Mu...
4,Luxor Temple,Luxor,https://egymonuments.com/storage/events/Septem...,https://egymonuments.com/details/LuxorTemple,Adult: EGP 500\nStudent: EGP 250,Adult: EGP 40\nStudent: EGP 20,All Days\nSummer Working Hours:\nfrom 06:00 am...,https://egymonuments.com/book-date/6,Free entry for children below 6 years. ...,https://www.google.com/maps/place/Luxor+Temple...
...,...,...,...,...,...,...,...,...,...,...
89,Kafr al-Sheikh Museum,kafr el-shiekh,https://egymonuments.com/storage/events/July20...,https://egymonuments.com/details/KafrElshaikh,Adult: EGP 220\nStudent: EGP 110,Adult: EGP 20\nStudent: EGP 10,Summer Working Hours:\nfrom 09:00 am Last Entr...,https://egymonuments.com/book-date/246,Free entry for children below 6 years. ...,https://www.google.com/maps/search/Kafr+al-She...
90,Tanis (San al-Hagar),Al-Sharqia,https://egymonuments.com/storage/events/Septem...,https://egymonuments.com/details/Tanis%28Sanal...,Adult: EGP 120\nStudent: EGP 60,Adult: EGP 10\nStudent: EGP 5,Summer Working Hours:\nfrom 09:00 am Last Entr...,https://egymonuments.com/book-date/249,Free entry for children below 6 years. ...,https://www.google.com/maps/search/Tanis+(San+...
91,Grand Egyptian Museum,Giza,https://egymonuments.com/storage/events/August...,https://visit-gem.com/,Free,Not Available,Not Available,NaN,Not Available,https://www.google.com/maps/search/Grand+Egypt...
92,Temple of Aghurmi in Siwa,Siwa,https://egymonuments.com/storage/events/Januar...,https://egymonuments.com/details/TempleofAghur...,Adult: EGP 120\nStudent: EGP 60,Adult: EGP 10\nStudent: EGP 5,All Days\nSummer Working Hours:\nfrom 09:00 am...,https://egymonuments.com/book-date/256,Free entry for children below 6 years. ...,https://www.google.com/maps/place/Amon+Temple/...


In [10]:
df.to_csv("egymonuments_tickets.csv", index=False)
